# Awaaz Relay — End-to-End System Notebook

**Multilingual AI copilot for Tamil Nadu frontline welfare workers**

This notebook walks through the complete Awaaz Relay pipeline:

| Step | Module | What it does |
|------|--------|--------------|
| 1 | Knowledge Base | 214 verified Tamil Nadu government welfare facts |
| 2 | Retrieval | Hybrid BM25 + semantic search |
| 3 | AI Orchestration | Gemini model chain → structured JSON |
| 4 | Safety Scoring | Confidence blend + escalation gates |
| 5 | End-to-End Demo | Multi-language, multi-scenario live runs |

> **No API key required** — all cells run with `DEMO_MODE=true` (mock responses).  
> To run against real Gemini: set `GOOGLE_API_KEY` in your environment before starting.

---
## 0 · Setup

In [ ]:
import sys, os, json, re, math, warnings
warnings.filterwarnings('ignore')

# ── resolve paths ────────────────────────────────────────────────────────────
REPO_ROOT    = os.path.abspath(os.path.join(os.getcwd()))
BACKEND_DIR  = os.path.join(REPO_ROOT, 'backend')
DATA_DIR     = os.path.join(REPO_ROOT, 'data')
sys.path.insert(0, BACKEND_DIR)

# ── demo mode: no real API key needed ────────────────────────────────────────
os.environ['DEMO_MODE']       = 'true'
os.environ['GOOGLE_API_KEY']  = 'demo-key-not-real'

print('Python :', sys.version.split()[0])
print('Backend:', BACKEND_DIR)
print('Data   :', DATA_DIR)

In [ ]:
# Install lightweight deps (only needed in fresh Colab/Kaggle envs)
import importlib
missing = [pkg for pkg in ['fastapi', 'pydantic', 'httpx'] if importlib.util.find_spec(pkg) is None]
if missing:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('Installed:', missing)
else:
    print('All dependencies already installed.')

---
## 1 · Knowledge Base Exploration

214 facts extracted from official Tamil Nadu government portals (2024–2026).  
Each fact carries a `rule_id`, `category`, source citation, and a `confidence_base` score.

In [ ]:
KB_PATH = os.path.join(DATA_DIR, 'knowledge_base.json')
with open(KB_PATH, 'r', encoding='utf-8') as f:
    kb_data = json.load(f)

facts = kb_data['facts']
print(f'Total facts: {len(facts)}')
print('\nSample fact:')
print(json.dumps(facts[0], indent=2, ensure_ascii=False))

In [ ]:
from collections import Counter

# ── category distribution ────────────────────────────────────────────────────
category_counts = Counter(f['category'] for f in facts)
sorted_cats = sorted(category_counts.items(), key=lambda x: -x[1])

print(f'{'Category':<25} {'Facts':>6}  Bar')
print('-' * 50)
for cat, count in sorted_cats:
    bar = '█' * count
    print(f'{cat:<25} {count:>6}  {bar}')

# ── confidence distribution ──────────────────────────────────────────────────
conf_values = [f['confidence_base'] for f in facts]
avg_conf = sum(conf_values) / len(conf_values)
high_conf = sum(1 for c in conf_values if c >= 0.9)
print(f'\nAverage confidence_base : {avg_conf:.3f}')
print(f'Facts with conf ≥ 0.90  : {high_conf} / {len(facts)}')

In [ ]:
# ── sample 3 facts per category ──────────────────────────────────────────────
from collections import defaultdict
by_cat = defaultdict(list)
for f in facts:
    by_cat[f['category']].append(f)

print('=== Sample facts by category ===\n')
for cat, cat_facts in list(by_cat.items())[:4]:
    print(f'── {cat.upper()} ({len(cat_facts)} facts) ──')
    for f in cat_facts[:2]:
        print(f"  [{f['rule_id']}] {f['text'][:110]}...")
    print()

---
## 2 · Language Detection

Script-range Unicode detection — no external NLP library needed, works offline.

In [ ]:
from input_processor import detect_language

test_phrases = [
    ('ta', 'விதவை ஓய்வூதியத்திற்கு எப்படி விண்ணப்பிக்கவும்?'),
    ('te', 'వితంతు పెన్షన్ దరఖాస్తు ఎలా చేయాలి?'),
    ('kn', 'ವಿಧವಾ ಪಿಂಚಣಿ ಅರ್ಜಿ ಹೇಗೆ ಸಲ್ಲಿಸಬೇಕು?'),
    ('hi', 'विधवा पेंशन के लिए आवेदन कैसे करें?'),
    ('en', 'How do I apply for widow pension?'),
    ('en', 'What documents are needed for old age pension?'),
]

lang_names = {'ta': 'Tamil', 'te': 'Telugu', 'kn': 'Kannada', 'hi': 'Hindi', 'en': 'English'}

print(f'{'Expected':<10} {'Detected':<10} {'Match':<6}  Query')
print('-' * 75)
correct = 0
for expected, phrase in test_phrases:
    detected = detect_language(phrase)
    match = '✓' if detected == expected else '✗'
    if detected == expected:
        correct += 1
    print(f'{lang_names[expected]:<10} {lang_names.get(detected, detected):<10} {match:<6}  {phrase[:55]}')

print(f'\nAccuracy: {correct}/{len(test_phrases)}')

---
## 3 · BM25 Retrieval

Keyword retrieval with TF-IDF scoring and category boosting — used as a fallback  
when embedding vectors are unavailable (no API key / first run).

In [ ]:
from retriever import bm25_retrieve
from schemas import CaseInput

queries = [
    ('en', 'What documents do I need for widow pension?'),
    ('en', 'My application was rejected, how do I appeal?'),
    ('en', 'What is the income limit for widow pension eligibility?'),
    ('ta', 'விண்ணப்பிக்க கடைசி தேதி என்ன?'),
]

for lang, query in queries:
    case = CaseInput(input_type='text', language=lang, query=query)
    result = bm25_retrieve(case, facts, top_k=3)

    print(f'Query: "{query}"')
    print(f'Retrieval score: {result.retrieval_score:.3f}  |  Method: {result.retrieval_method}')
    for i, fact in enumerate(result.retrieved_facts, 1):
        print(f'  {i}. [{fact.rule_id}] {fact.category.upper()} (rel={fact.relevance_score:.3f})')
        print(f'     {fact.text[:100]}...')
    print()

---
## 4 · AI Orchestration (Demo Mode)

The orchestrator builds a structured prompt from the retrieved facts and sends it to  
the Gemini model chain. In demo mode, realistic mock responses are returned instantly.

In [ ]:
from gemma_orchestrator import orchestrate, _detect_mock_scenario
from schemas import CaseInput, RetrievalResult, RetrievedFact

def make_mock_retrieval(query: str, lang: str = 'en') -> RetrievalResult:
    """Build a minimal RetrievalResult for demo purposes."""
    sample_facts = [
        RetrievedFact(
            rule_id='PENSION_001', category='eligibility',
            text='An applicant must be a widow to be eligible for Tamil Nadu Widow Pension.',
            source='TN Social Welfare Dept', confidence_base=0.98, relevance_score=0.85
        ),
        RetrievedFact(
            rule_id='PENSION_003', category='eligibility',
            text='The applicant must not earn more than ₹2,40,000 per annum.',
            source='TN Social Welfare Dept', confidence_base=0.97, relevance_score=0.82
        ),
        RetrievedFact(
            rule_id='PENSION_008', category='documents',
            text='Required documents: FORM 1-W, death certificate, Aadhaar/Voter ID, bank passbook, ration card, 2 photos.',
            source='TN District Portal', confidence_base=0.96, relevance_score=0.79
        ),
    ]
    return RetrievalResult(
        retrieved_facts=sample_facts,
        retrieval_score=0.82,
        query_used=query,
        retrieval_method='bm25'
    )

# Test scenario detection
scenario_queries = [
    'What documents do I need?',
    'Is June 30 the last date?',
    'My application was rejected',
    'She is 62 years old and a widow',
    'She is ill and cannot walk to the office',
]

print('Scenario detection:')
print(f'{'Query':<45} Scenario')
print('-' * 60)
for q in scenario_queries:
    case = CaseInput(input_type='text', language='en', query=q)
    scenario = _detect_mock_scenario(case)
    print(f'{q:<45} → {scenario}')

In [ ]:
# ── run orchestrator in demo mode ────────────────────────────────────────────
query  = 'The applicant earns Rs 3500 per month. Does she qualify for widow pension?'
lang   = 'en'

case      = CaseInput(input_type='text', language=lang, query=query)
retrieval = make_mock_retrieval(query, lang)
response  = orchestrate(case, retrieval, demo_mode=True)

print(f'Domain           : {response.domain}')
print(f'Confidence       : {response.confidence_percent}%  [{response.confidence_band}]')
print(f'Escalation needed: {response.escalation_needed}')
print()
print('Worker Guidance (checklist):')
for i, step in enumerate(response.worker_guidance, 1):
    print(f'  {i}. {step}')
print()
print('Citizen Guidance (plain language):')
print(f'  {response.citizen_guidance}')
print()
print('Evidence used:', response.evidence_used)

---
## 5 · Safety Scorer

Blends retrieval quality (60%) and LLM self-estimate (40%) into a final confidence score.  
Queries below 40% confidence are suppressed — helpline is shown instead.

In [ ]:
import safety_scorer

# Run safety scorer on the response from Section 4
safety = safety_scorer.score(response, retrieval)

print(f'Confidence        : {safety.confidence_percent}%')
print(f'Band              : {safety.confidence_band}')
print(f'Safe to answer    : {safety.safe_to_answer}')
print(f'Show warning      : {safety.show_warning}')
print(f'Escalate          : {safety.escalate_immediately}')
print()
print('Evidence panel:')
for f in safety.evidence_panel:
    print(f'  [{f.rule_id}] {f.category} — {f.text[:80]}...')
print()
print('Disclaimer:')
print(f'  {safety.disclaimer}')

In [ ]:
# ── confidence formula demonstration ─────────────────────────────────────────
print('Confidence formula: (retrieval_score × 0.6) + (gemini_self_estimate × 0.4)')
print()

scenarios_conf = [
    ('High — clear eligibility query',  0.85, 0.88),
    ('Medium — missing document query', 0.60, 0.55),
    ('Low — out-of-scope medical query', 0.20, 0.25),
]

print(f'{'Scenario':<38} {'Ret':>6}  {'LLM':>6}  {'Blend':>6}  Band')
print('-' * 70)
for name, ret, llm in scenarios_conf:
    blend = round(ret * 0.6 + llm * 0.4, 3)
    pct   = int(blend * 100)
    band  = 'high' if pct >= 70 else 'medium' if pct >= 40 else 'low'
    gate  = '✓ Answer' if pct >= 40 else '✗ Suppress → helpline'
    print(f'{name:<38} {ret:>6.2f}  {llm:>6.2f}  {pct:>5}%  {band:<7} {gate}')

---
## 6 · End-to-End Pipeline — Multiple Scenarios

Simulates 6 real community worker queries covering the main welfare scenarios.

In [ ]:
from retriever import bm25_retrieve

def run_pipeline(query: str, lang: str = 'en') -> dict:
    """Run the full Awaaz Relay pipeline and return a summary dict."""
    case      = CaseInput(input_type='text', language=lang, query=query)
    retrieval = bm25_retrieve(case, facts, top_k=5)
    gemma_out = orchestrate(case, retrieval, demo_mode=True)
    safety    = safety_scorer.score(gemma_out, retrieval)
    return {
        'query'      : query,
        'language'   : lang,
        'domain'     : gemma_out.domain,
        'confidence' : safety.confidence_percent,
        'band'       : safety.confidence_band,
        'safe'       : safety.safe_to_answer,
        'guidance_0' : gemma_out.worker_guidance[0] if gemma_out.worker_guidance else '',
        'escalate'   : safety.escalate_immediately,
        'regional'   : gemma_out.regional_summary,
    }

scenarios = [
    ('en', 'The widow earns Rs 3500 per month. Does she qualify for pension?'),
    ('en', 'What documents are needed to apply for widow pension?'),
    ('en', 'Is June 30 the last date to apply? What happens if she misses it?'),
    ('en', 'Her application was rejected. How do I file an appeal?'),
    ('en', 'The applicant is 65 years old. Which pension scheme applies?'),
    ('en', 'She is ill and asks about her health condition for the form.'),
]

results = [run_pipeline(q, lang) for lang, q in scenarios]

# ── summary table ─────────────────────────────────────────────────────────────
print(f'{'#':<3} {'Conf':>5}  {'Band':<8} {'Safe':<6} {'Domain':<25}  Query')
print('─' * 95)
for i, r in enumerate(results, 1):
    safe_icon = '✓' if r['safe'] else '✗'
    esc_icon  = ' [ESCALATE]' if r['escalate'] else ''
    query_short = r['query'][:55]
    print(f"{i:<3} {r['confidence']:>4}%  {r['band']:<8} {safe_icon:<6} {r['domain']:<25}  {query_short}{esc_icon}")

In [ ]:
# ── detailed output for the eligibility scenario ──────────────────────────────
r = results[0]
case      = CaseInput(input_type='text', language=r['language'], query=r['query'])
retrieval = bm25_retrieve(case, facts, top_k=5)
gemma_out = orchestrate(case, retrieval, demo_mode=True)

print('=' * 70)
print('SCENARIO: Eligibility check (income qualification)')
print('=' * 70)
print(f'Query    : {r["query"]}')
print(f'Confidence: {r["confidence"]}%  [{r["band"]}]')
print()
print('WORKER CHECKLIST:')
for i, step in enumerate(gemma_out.worker_guidance, 1):
    print(f'  {i}. {step}')
print()
print('CITIZEN GUIDANCE:')
print(f'  {gemma_out.citizen_guidance}')

In [ ]:
# ── detailed output for the low-confidence (out-of-scope) scenario ────────────
r = results[5]
case      = CaseInput(input_type='text', language=r['language'], query=r['query'])
retrieval = bm25_retrieve(case, facts, top_k=5)
gemma_out = orchestrate(case, retrieval, demo_mode=True)
safety    = safety_scorer.score(gemma_out, retrieval)

print('=' * 70)
print('SCENARIO: Out-of-scope (medical query) — safety gate activated')
print('=' * 70)
print(f'Query       : {r["query"]}')
print(f'Confidence  : {r["confidence"]}%  [{r["band"]}]')
print(f'Safe to ans : {safety.safe_to_answer}  ← answer SUPPRESSED')
print(f'Escalate    : {safety.escalate_immediately}')
print()
print('DISCLAIMER / ESCALATION:')
print(f'  {safety.disclaimer}')

---
## 7 · Multi-Language Output

The same query in all 5 supported languages — demonstrating that the regional summary  
is returned in the citizen's own language.

In [ ]:
LANG_NAMES = {'ta': 'Tamil', 'te': 'Telugu', 'kn': 'Kannada', 'hi': 'Hindi', 'en': 'English'}

# Same eligibility question in all 5 languages
multilang_queries = [
    ('ta', 'விண்ணப்பிக்க ஜூன் 30 கடைசி தேதியா?'),
    ('te', 'జూన్ 30 చివరి తేదీ అయినప్పటికీ దరఖాస్తు చేయవచ్చా?'),
    ('kn', 'ಜೂನ್ 30 ಕೊನೆಯ ದಿನ. ಈಗ ಅರ್ಜಿ ಸಲ್ಲಿಸಬಹುದೇ?'),
    ('hi', '30 जून की अंतिम तारीख है, क्या अभी भी आवेदन हो सकता है?'),
    ('en', 'Is June 30 the deadline? What if documents are not ready yet?'),
]

print('Multi-language deadline query — regional summaries:\n')
for lang, query in multilang_queries:
    case      = CaseInput(input_type='text', language=lang, query=query)
    retrieval = bm25_retrieve(case, facts, top_k=5)
    gemma_out = orchestrate(case, retrieval, demo_mode=True)

    print(f'── {LANG_NAMES[lang]} ({lang}) ──')
    print(f'  Query   : {query}')
    if gemma_out.regional_summary:
        print(f'  Summary : {gemma_out.regional_summary}')
    else:
        print(f'  Summary : {gemma_out.citizen_guidance[:180]}...')
    print()

---
## 8 · FastAPI Endpoint Test

Uses the ASGI `TestClient` — no server needed, no ports opened.

In [ ]:
import os, sys
os.environ['DEMO_MODE'] = 'true'
os.environ.setdefault('GOOGLE_API_KEY', 'demo-key-not-real')

# The TestClient import triggers FastAPI lifespan
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

# ── /health ───────────────────────────────────────────────────────────────────
r = client.get('/health')
print('/health:', r.status_code)
print(json.dumps(r.json(), indent=2))

In [ ]:
# ── /kb/stats ─────────────────────────────────────────────────────────────────
r = client.get('/kb/stats')
print('/kb/stats:', r.status_code)
print(json.dumps(r.json(), indent=2))

In [ ]:
# ── POST /analyze (text query) ────────────────────────────────────────────────
payload = {
    'input_type' : 'text',
    'language'   : 'en',
    'text'       : 'The widow earns Rs 3500 per month. Does she qualify for the pension?',
}
r = client.post('/analyze', data=payload)
print('/analyze:', r.status_code)
data = r.json()

print(f"\nCase ID     : {data['case_id']}")
print(f"Language    : {data['language']}")
print(f"Confidence  : {data['safety']['confidence_percent']}%  [{data['safety']['confidence_band']}]")
print(f"Domain      : {data['gemma']['domain']}")
print(f"Safe to ans : {data['safety']['safe_to_answer']}")
print(f"Processing  : {data['processing_time_ms']} ms")
print()
print('Worker guidance (first 3 steps):')
for i, step in enumerate(data['gemma']['worker_guidance'][:3], 1):
    print(f'  {i}. {step}')
print()
print('Citizen guidance:')
print(f"  {data['gemma']['citizen_guidance'][:300]}...")

In [ ]:
# ── POST /analyze (Tamil language) ───────────────────────────────────────────
payload = {
    'input_type' : 'text',
    'language'   : 'ta',
    'text'       : 'விதவை ஓய்வூதியம் பெற ஜூன் 30 கடைசி தேதியா?',
}
r = client.post('/analyze', data=payload)
print('Tamil query:', r.status_code)
data = r.json()
print(f"Confidence  : {data['safety']['confidence_percent']}%  [{data['safety']['confidence_band']}]")
print()
print('Regional summary (Tamil):')
regional = data['gemma'].get('regional_summary') or data['gemma'].get('tamil_summary')
print(f'  {regional}')

In [ ]:
# ── POST /feedback ────────────────────────────────────────────────────────────
import uuid
fb = client.post('/feedback', json={
    'case_id'           : str(uuid.uuid4()),
    'rating'            : 'up',
    'query_summary'     : 'Eligibility check for widow pension',
    'confidence_percent': 88,
})
print('/feedback:', fb.status_code, fb.json())

---
## 9 · Confidence Score Visualization

ASCII chart showing how the confidence blending works across all 7 mock scenarios.

In [ ]:
from gemma_orchestrator import MOCK_RESPONSES

print('Awaaz Relay — Confidence Scores Across Demo Scenarios')
print('=' * 60)
print(f'  Confidence = (Retrieval × 0.6) + (LLM estimate × 0.4)')
print()

# Using representative retrieval scores for each scenario
retrieval_scores = {
    'high'    : 0.85,
    'medium'  : 0.55,
    'low'     : 0.18,
    'deadline': 0.88,
    'process' : 0.80,
    'old_age' : 0.83,
    'appeal'  : 0.78,
}

BAND_COLOR = {'high': '▓', 'medium': '░', 'low': '·'}
BAND_LABEL = {'high': 'HIGH   ✓', 'medium': 'MEDIUM ⚠', 'low': 'LOW    ✗ (suppressed)'}

print(f"{'Scenario':<12} {'LLM':>4}  {'Ret':>4}  {'Final':>6}  {'':45} Band")
print('─' * 85)

for scenario, mock in MOCK_RESPONSES.items():
    ret   = retrieval_scores.get(scenario, 0.75)
    llm   = mock.confidence
    blend = round(ret * 0.6 + llm * 0.4, 3)
    pct   = int(blend * 100)
    band  = 'high' if pct >= 70 else 'medium' if pct >= 40 else 'low'
    bar   = BAND_COLOR[band] * pct + ' ' * (100 - pct)
    bar   = bar[:45]
    print(f"{scenario:<12} {llm:>4.2f}  {ret:>4.2f}  {pct:>4}%   [{bar}] {BAND_LABEL[band]}")

print()
print('Legend: ▓ = high (≥70%)   ░ = medium (40-69%)   · = low (<40%)')

---
## 10 · Multi-Turn Conversation Context

Awaaz Relay passes up to 8 prior turns to Gemini for follow-up reasoning.

In [ ]:
# Simulate a 3-turn conversation about widow pension
conversation_history = [
    {
        'query'            : 'Does the applicant qualify for widow pension?',
        'domain'           : 'pension_eligibility',
        'confidence_percent': 88,
        'guidance_summary' : 'Applicant qualifies on income (Rs 3500/month < Rs 2.4L limit)',
    },
    {
        'query'            : 'She does not have Aadhaar. Can she still apply?',
        'domain'           : 'documents_required',
        'confidence_percent': 72,
        'guidance_summary' : 'Voter ID or Ration Card accepted as alternative ID',
    },
]

followup = 'What is the deadline and how do we track the application after submitting?'
case      = CaseInput(input_type='text', language='en', query=followup)
retrieval = bm25_retrieve(case, facts, top_k=5)
response  = orchestrate(case, retrieval, demo_mode=True,
                        conversation_history=conversation_history)

print('Multi-turn conversation follow-up')
print('─' * 50)
print(f'Turn 1: Does she qualify?              → {conversation_history[0]["domain"]} ({conversation_history[0]["confidence_percent"]}%)')
print(f'Turn 2: No Aadhaar, can she apply?     → {conversation_history[1]["domain"]} ({conversation_history[1]["confidence_percent"]}%)')
print(f'Turn 3: {followup}')
print()
print(f'Response domain     : {response.domain}')
print(f'Confidence          : {response.confidence_percent}%')
print()
print('Worker guidance:')
for i, step in enumerate(response.worker_guidance[:4], 1):
    print(f'  {i}. {step}')

---
## 11 · Welfare Scheme Coverage

Summary of the 12 schemes in the knowledge base.

In [ ]:
schemes = [
    ('Widow Pension (FORM 1-W)',             'Widows 18–65 yrs, income < ₹2.4L/yr',     '₹1,000/month'),
    ('Old Age Pension (IGNOAPS)',            'Citizens 60+ yrs, BPL',                    '₹1,000/month'),
    ('Disability Pension',                  '40%+ disability, income < ₹2.4L/yr',       '₹1,000/month'),
    ('Destitute Widow Pension',             'Widows with no income at all',             '₹1,500/month'),
    ('Deserted Women Pension',              'Women abandoned by spouse, no income',     '₹1,000/month'),
    ('Transgender Welfare',                 'Transgender persons, 18+ yrs',             '₹1,000/month'),
    ('Girl Child Welfare Scheme',           'BPL families, girl child registered',      'One-time grant'),
    ('Unmarried Women Pension',             'Women above 30 yrs, unmarried, BPL',       '₹1,000/month'),
    ('Agricultural Labourer Pension',       'Reg. agri. labourers, 60+ yrs',            '₹1,000/month'),
    ('Differently-Abled Pension',           'Severe disability cert., BPL',             '₹1,500/month'),
    ('Destitute Fishermen Pension',         'Reg. fishermen, 60+ yrs',                  '₹1,000/month'),
    ('Leprosy Affected Pension',            'Leprosy-affected citizens, any age',       '₹1,000/month'),
]

print(f'{'Scheme':<38} {'Eligibility':<40} Benefit')
print('─' * 95)
for name, elig, benefit in schemes:
    print(f'{name:<38} {elig:<40} {benefit}')

print(f'\nTotal schemes supported: {len(schemes)}')
print(f'Total KB facts         : {len(facts)}')
print(f'Languages supported    : Tamil, Telugu, Kannada, Hindi, English')

---
## 12 · Impact Summary

Quantifying the gap Awaaz Relay is designed to close.

In [ ]:
print('''
╔══════════════════════════════════════════════════════════════════╗
║              AWAAZ RELAY — IMPACT SNAPSHOT                      ║
╠══════════════════════════════════════════════════════════════════╣
║  Problem scale                                                  ║
║  • 10 million+ citizens eligible for TN welfare pensions        ║
║  • Estimated 30–40% enrollment gap (eligible but not enrolled)  ║
║  • Rejected applications → 8-month wait for re-application      ║
╠══════════════════════════════════════════════════════════════════╣
║  System specs                                                   ║
║  • 214 verified KB facts  |  12 welfare schemes                 ║
║  • 5 languages            |  3 input modes (text/image/voice)   ║
║  • Hybrid retrieval       |  3-model Gemini chain               ║
║  • 70 automated tests     |  CI/CD via GitHub Actions           ║
╠══════════════════════════════════════════════════════════════════╣
║  Safety design                                                  ║
║  • Confidence < 40%  → answer suppressed, helpline shown        ║
║  • No answer without retrieval grounding (zero hallucination)   ║
║  • Every response shows source rule IDs + confidence score      ║
║  • Disclaimer on every response: not legal advice               ║
║  • Rate limit: 20 req/min per IP (SQLite-backed)                ║
╠══════════════════════════════════════════════════════════════════╣
║  Helpline  1800-425-1700  (toll-free, Mon–Sat 9 AM–5 PM)       ║
╚══════════════════════════════════════════════════════════════════╝
''')

print('Run this notebook top-to-bottom to see the full pipeline in action.')
print('Set GOOGLE_API_KEY in your env (or .env file) to use live Gemini.')